# Анализ датасета [Russian Financial News](https://www.kaggle.com/datasets/kkhubiev/russian-financial-news)

In [1]:
import json
import re

def parse_tickers_description_full(file_path):
    
    tickers_info = {}
    name_to_ticker = {}
    
    # Читаем файл
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read().strip()
    
    # Парсим JSON
    try:
        data = json.loads(content)
        
        if isinstance(data, dict):
            for ticker, description in data.items():
                # Сохраняем информацию
                tickers_info[ticker] = {
                    'ticker': ticker,
                    'description': description[:200] if len(description) > 200 else description
                }
                
                # Извлекаем название компании
                company_name = extract_company_name_simple(description)
                if company_name:
                    tickers_info[ticker]['name'] = company_name
                    name_to_ticker[company_name] = ticker
                    name_to_ticker[company_name.upper()] = ticker
                
                # Сохраняем сам тикер
                name_to_ticker[ticker] = ticker
                name_to_ticker[ticker.upper()] = ticker
                name_to_ticker[ticker.lower()] = ticker
                
    except json.JSONDecodeError as e:
        print(f"JSON decode error: {e}")
        # Пробуем читать построчно
        lines = content.split('\n')
        for line in lines:
            if line.strip():
                try:
                    data = json.loads(line)
                    if isinstance(data, dict):
                        for ticker, description in data.items():
                            tickers_info[ticker] = {
                                'ticker': ticker,
                                'description': description[:200] if len(description) > 200 else description
                            }
                            company_name = extract_company_name_simple(description)
                            if company_name:
                                tickers_info[ticker]['name'] = company_name
                                name_to_ticker[company_name] = ticker
                                name_to_ticker[company_name.upper()] = ticker
                            name_to_ticker[ticker] = ticker
                            name_to_ticker[ticker.upper()] = ticker
                except:
                    continue
    
    return tickers_info, name_to_ticker

def extract_company_name_simple(description):
    if not description:
        return None
    
    # Очищаем от escape последовательностей
    desc_clean = description
    # Простая замена для русских букв
    replacements = {
        '\\u0430': 'а', '\\u0431': 'б', '\\u0432': 'в', '\\u0433': 'г', '\\u0434': 'д',
        '\\u0435': 'е', '\\u0436': 'ж', '\\u0437': 'з', '\\u0438': 'и', '\\u0439': 'й',
        '\\u043a': 'к', '\\u043b': 'л', '\\u043c': 'м', '\\u043d': 'н', '\\u043e': 'о',
        '\\u043f': 'п', '\\u0440': 'р', '\\u0441': 'с', '\\u0442': 'т', '\\u0443': 'у',
        '\\u0444': 'ф', '\\u0445': 'х', '\\u0446': 'ц', '\\u0447': 'ч', '\\u0448': 'ш',
        '\\u0449': 'щ', '\\u044a': 'ъ', '\\u044b': 'ы', '\\u044c': 'ь', '\\u044d': 'э',
        '\\u044e': 'ю', '\\u044f': 'я', '\\u2014': '-', '\\u00ab': '"', '\\u00bb': '"'
    }
    
    for code, char in replacements.items():
        desc_clean = desc_clean.replace(code, char)
    
    # Извлекаем название до запятой, скобки или точки
    separators = [',', '(', '—', '-', '«', '»']
    name = desc_clean
    
    for sep in separators:
        if sep in name:
            name = name.split(sep)[0]
            break
    
    # Очищаем от лишних символов
    name = name.strip().strip('"').strip("'")
    name = re.sub(r'^(ПАО|АО|ОАО|ЗАО|ООО)\s+', '', name)
    
    words = name.split()
    if len(words) > 5:
        name = ' '.join(words[:4])
    
    return name if len(name) > 2 else None

In [2]:
file_path = "./RussianFinancialNews/tickers_description.json"
        
tickers_info, name_to_ticker = parse_tickers_description_full(file_path)
    
print(f"Всего тикеров: {len(tickers_info)}")
print(f"Уникальных названий для маппинга: {len(name_to_ticker)}")
        
output_file = "./data/ticker_mapping_full.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(name_to_ticker, f, ensure_ascii=False, indent=2)
    
tickers_file = "./data/tickers_info.json"
with open(tickers_file, 'w', encoding='utf-8') as f:
    json.dump(tickers_info, f, ensure_ascii=False, indent=2)


Всего тикеров: 238
Уникальных названий для маппинга: 862


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import json
from collections import Counter

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

class RussianFinancialNewsAnalyzer:
    def __init__(self, data_path):
        self.data_path = Path(data_path)
        self.news_df = None
        self.tickers_desc = {}
        self.name_to_ticker = {}
        self.candles_data = {}
        self.analysis_results = {}
        self.labeled_data = {}
        self.available_tickers = set()
        
    def load_data(self):
        # Загрузка новостей
        news_file = self.data_path / 'news_collection.parquet'
        if news_file.exists():
            self.news_df = pd.read_parquet(news_file)
            print(f"Новости загружены: {len(self.news_df):,} записей")            
        else:
            print(f"Файл новостей не найден: {news_file}")
            return self
            
        # Загрузка маппинга названий к тикерам
        mapping_file = self.data_path.parent / 'data/ticker_mapping_full.json'
        if mapping_file.exists():
            self._load_ticker_mapping(mapping_file)
            print(f"Маппинг тикеров загружен: {len(self.name_to_ticker)} записей")
        else:
            print(f"Файл маппинга не найден: {mapping_file}")

            
        # Загрузка свечных данных
        self._load_candles_data()
        
        # Загрузка размеченных данных
        self._load_labeled_data()
        
        # Сопоставляем теги с биржевыми тикерами
        self._map_tags_to_tickers()
        
        return self
        
    def _load_ticker_mapping(self, file_path):
        """Загрузка маппинга названий к тикерам"""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                self.name_to_ticker = json.load(f)
        except Exception as e:
            print(f"Ошибка загрузки маппинга: {e}")
            self._create_basic_mapping()
    
    def _load_candles_data(self):
        """Загрузка свечных данных"""
        timeframes = ['1h', '1d', '10m']
        
        for tf in timeframes:
            tf_dir = self.data_path / 'candles_data' / tf
            if not tf_dir.exists():
                continue
            
            close_file = tf_dir / 'close.csv'
            if not close_file.exists():
                continue
            
            print(f"Загрузка данных для таймфрейма {tf}...")
            
            close_df = pd.read_csv(close_file)
            open_df = pd.read_csv(tf_dir / 'open.csv') if (tf_dir / 'open.csv').exists() else None
            high_df = pd.read_csv(tf_dir / 'high.csv') if (tf_dir / 'high.csv').exists() else None
            low_df = pd.read_csv(tf_dir / 'low.csv') if (tf_dir / 'low.csv').exists() else None
            
            date_col = close_df.columns[0]
            tickers = close_df.columns[1:]
            
            print(f"Найдено {len(tickers)} тикеров")
            
            loaded = 0
            for ticker in tickers:
                try:
                    df = pd.DataFrame()
                    df['begin'] = pd.to_datetime(close_df[date_col])
                    df['close'] = close_df[ticker]
                    
                    if open_df is not None and ticker in open_df.columns:
                        df['open'] = open_df[ticker]
                    if high_df is not None and ticker in high_df.columns:
                        df['high'] = high_df[ticker]
                    if low_df is not None and ticker in low_df.columns:
                        df['low'] = low_df[ticker]
                    
                    df = df.dropna()
                    
                    if len(df) > 0:
                        self.candles_data[ticker] = df
                        self.available_tickers.add(ticker)
                        loaded += 1
                        
                except Exception as e:
                    continue
            
            print(f"Загружено {loaded} тикеров для {tf}")
            if loaded > 0:
                break
        
        if not self.candles_data:
            print(f"Свечные данные не найдены")
    
    def _load_labeled_data(self):
        """Загрузка размеченных данных"""
        news_desc_dir = self.data_path / 'news_descriptions'
        
        labeled_file = news_desc_dir / 'news_collection_old.parquet'
        if labeled_file.exists():
            self.labeled_data['old'] = pd.read_parquet(labeled_file)
            print(f"Размеченные данные (old): {len(self.labeled_data['old']):,} записей")
        
        # Загрузка GPT4o
        gpt4o_file = news_desc_dir / 'news_descriptions_GPT4o.json'
        if gpt4o_file.exists():
            self.labeled_data['gpt4o'] = self._load_labeled_json(gpt4o_file)
            print(f"GPT4o разметка: {len(self.labeled_data['gpt4o']):,} записей")
        
        # Загрузка LLama3
        llama3_file = news_desc_dir / 'news_description_LLama3_8b.json'
        if llama3_file.exists():
            self.labeled_data['llama3'] = self._load_labeled_json(llama3_file)
            print(f"LLama3 разметка: {len(self.labeled_data['llama3']):,} записей")
    
    def _load_labeled_json(self, file_path):
        """Загрузка JSON с разметкой"""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            records = []
            for key, value in data.items():
                if isinstance(value, dict):
                    if 'description' in value:
                        record = value['description'].copy()
                    else:
                        record = value.copy()
                    record['index'] = int(key)
                    records.append(record)
            
            return pd.DataFrame(records)
        except Exception as e:
            print(f"Ошибка загрузки {file_path}: {e}")
            return pd.DataFrame()
    
    def _map_tags_to_tickers(self):
        """Сопоставление тегов с биржевыми тикерами"""
        if self.news_df is None:
            return
        
        # Функция для сопоставления
        def map_to_ticker(tag):
            if not tag:
                return None
            
            # Прямое сопоставление
            if tag in self.name_to_ticker:
                return self.name_to_ticker[tag]
            
            # Поиск по частичному совпадению (убираем пробелы и приводим к верхнему регистру)
            tag_clean = tag.replace(' ', '').replace('-', '').upper()
            for name, ticker in self.name_to_ticker.items():
                name_clean = name.replace(' ', '').replace('-', '').upper()
                if tag_clean == name_clean or tag_clean in name_clean or name_clean in tag_clean:
                    return ticker
            
            return None
        
        # Преобразуем теги
        def convert_tags(tags):
            if not tags:
                return []
            result = []
            for tag in tags:
                ticker = map_to_ticker(tag)
                if ticker:
                    result.append(ticker)
            return list(set(result))  # уникальные тикеры
        
        self.news_df['tickers_mapped'] = self.news_df['tags'].apply(convert_tags)
        self.news_df['has_tickers'] = self.news_df['tickers_mapped'].apply(len) > 0
        
        # Подсчитываем статистику
        all_mapped_tickers = []
        for tickers in self.news_df['tickers_mapped']:
            all_mapped_tickers.extend(tickers)
        
        self.mapped_ticker_counts = Counter(all_mapped_tickers)
        
        print(f"СОПОСТАВЛЕНИЕ ТИКЕРОВ:")
        print(f"Новости с сопоставленными тикерами: {self.news_df['has_tickers'].sum():,} ({self.news_df['has_tickers'].mean()*100:.1f}%)")
        print(f"Уникальных сопоставленных тикеров: {len(self.mapped_ticker_counts)}")
        
        # Показываем топ-20
        print(f"ТОП-20 СОПОСТАВЛЕННЫХ ТИКЕРОВ:")
        for ticker, count in self.mapped_ticker_counts.most_common(20):
            has_price = "yes" if ticker in self.available_tickers else "no"
            print(f"   {has_price} {ticker}: {count:,}")
    
    def analyze_news(self):
        """Анализ новостных данных"""
        if self.news_df is None:
            print("Данные новостей не загружены")
            return
        
        df = self.news_df.copy()
        
        print(f"БАЗОВАЯ СТАТИСТИКА:")
        print(f"Всего новостей: {len(df):,}")
        print(f"Уникальных заголовков: {df['title'].nunique():,}")
        print(f"Новости с тикерами (после маппинга): {df['has_tickers'].sum():,} ({df['has_tickers'].mean()*100:.1f}%)")
        
        # Анализ источников
        print(f"АНАЛИЗ ИСТОЧНИКОВ:")
        source_stats = df['source'].value_counts()
        for source, count in source_stats.items():
            print(f"{source}: {count:,} ({count/len(df)*100:.1f}%)")
        
        # Временной анализ
        print(f"ВРЕМЕННОЙ АНАЛИЗ:")
        df['datetime'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
        df = df.dropna(subset=['datetime'])
        
        print(f"Период: {df['datetime'].min()} -> {df['datetime'].max()}")
        
        # Дни недели
        df['day_of_week'] = df['datetime'].dt.day_name()
        day_counts = df['day_of_week'].value_counts()
        print(f"Распределение по дням недели:")
        days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        for day in days_order:
            if day in day_counts:
                print(f"   {day}: {day_counts[day]:,} ({day_counts[day]/len(df)*100:.1f}%)")
        
        # Часы
        df['hour'] = df['datetime'].dt.hour
        print(f"Пик публикаций: {df['hour'].mode()[0]}:00")
        
        # Длина текстов
        print(f"АНАЛИЗ ДЛИНЫ ТЕКСТОВ:")
        df['title_len'] = df['title'].str.len()
        df['body_len'] = df['body'].fillna('').str.len()
        
        print(f"Заголовок - среднее: {df['title_len'].mean():.0f}, медиана: {df['title_len'].median():.0f}")
        print(f"Тело - среднее: {df['body_len'].mean():.0f}, медиана: {df['body_len'].median():.0f}")
    
    def analyze_candles_availability(self):
        """Анализ доступности ценовых данных"""        
        if not hasattr(self, 'mapped_ticker_counts'):
            print("Нет данных о сопоставленных тикерах")
            return
        
        # Тикеры из новостей после маппинга
        news_tickers = set(self.mapped_ticker_counts.keys())
        
        # Тикеры с ценами
        available = news_tickers & self.available_tickers
        not_available = news_tickers - self.available_tickers
        
        print(f"СТАТИСТИКА:")
        print(f"Уникальных тикеров в новостях: {len(news_tickers):,}")
        print(f"Тикеров с ценовыми данными: {len(available):,} ({len(available)/len(news_tickers)*100:.1f}%)")
        
        # Количество новостей с доступными ценами
        news_with_price = 0
        for ticker in available:
            news_with_price += self.mapped_ticker_counts[ticker]
        
        total_news_with_tickers = sum(self.mapped_ticker_counts.values())
        print(f"Новостей с тикерами: {total_news_with_tickers:,}")
        print(f"Новостей с доступными ценами: {news_with_price:,} ({news_with_price/total_news_with_tickers*100:.1f}%)")
        
        # Топ тикеров с ценами
        top_with_prices = [(ticker, self.mapped_ticker_counts[ticker]) 
                          for ticker in available 
                          if self.mapped_ticker_counts[ticker] > 0]
        top_with_prices.sort(key=lambda x: x[1], reverse=True)
        
        print(f"ТОП-20 ТИКЕРОВ С ЦЕНОВЫМИ ДАННЫМИ:")
        for ticker, count in top_with_prices[:20]:
            print(f"   {ticker}: {count:,} новостей")
        
        self.analysis_results['price_availability'] = {
            'total_tickers_in_news': len(news_tickers),
            'tickers_with_price': len(available),
            'news_with_price': news_with_price,
            'top_tickers_with_price': top_with_prices[:20]
        }
        
        return available, not_available
    
    def analyze_labeled_data(self):
        """Анализ размеченных данных"""        
        for model_name, df in self.labeled_data.items():
            if len(df) == 0:
                continue
                
            print(f"{model_name.upper()} РАЗМЕТКА:")
            print(f"Всего записей: {len(df):,}")
            
            # Анализ тональности
            if 'sentiment_score' in df.columns:
                sentiment = df['sentiment_score'].dropna()
                if len(sentiment) > 0:
                    print(f"АНАЛИЗ ТОНАЛЬНОСТИ:")
                    print(f"   Среднее: {sentiment.mean():.3f}")
                    print(f"   Медиана: {sentiment.median():.3f}")
                    
                    positive = (sentiment > 0.2).sum()
                    negative = (sentiment < -0.2).sum()
                    neutral = ((sentiment >= -0.2) & (sentiment <= 0.2)).sum()
                    
                    print(f"   Позитивные: {positive} ({positive/len(sentiment)*100:.1f}%)")
                    print(f"   Негативные: {negative} ({negative/len(sentiment)*100:.1f}%)")
                    print(f"   Нейтральные: {neutral} ({neutral/len(sentiment)*100:.1f}%)")
    
    def generate_report(self):
        """Генерация отчета"""        
        if hasattr(self, 'mapped_ticker_counts'):
            total_news_with_tickers = sum(self.mapped_ticker_counts.values())
            print(f"ДАННЫЕ ДЛЯ РАЗМЕТКИ:")
            print(f"Всего новостей с тикерами: {total_news_with_tickers:,}")
        
        if 'price_availability' in self.analysis_results:
            pa = self.analysis_results['price_availability']
            print(f"ЦЕНОВЫЕ ДАННЫЕ:")
            print(f"Тикеров с ценами: {pa['tickers_with_price']}")
            print(f"Новостей с ценами: {pa['news_with_price']:,}")

In [4]:
analyzer = RussianFinancialNewsAnalyzer("/home/mitchell/dev/hse/HSE_LLM_2026/RussianFinancialNews")
analyzer.load_data()
None

Новости загружены: 91,955 записей
Маппинг тикеров загружен: 862 записей
Загрузка данных для таймфрейма 1h...
Найдено 252 тикеров
Загружено 250 тикеров для 1h
Размеченные данные (old): 79,705 записей
GPT4o разметка: 15,000 записей
LLama3 разметка: 24,398 записей
СОПОСТАВЛЕНИЕ ТИКЕРОВ:
Новости с сопоставленными тикерами: 74,164 (80.7%)
Уникальных сопоставленных тикеров: 24
ТОП-20 СОПОСТАВЛЕННЫХ ТИКЕРОВ:
   yes ABIO: 73,475
   yes AFKS: 58,974
   yes AFLT: 47,794
   yes AKRN: 42,989
   no BLNG: 40,687
   yes ABRD: 32,222
   yes CARM: 10,562
   yes APRI: 10,433
   yes BANE: 6,893
   yes AQUA: 5,895
   yes BRZL: 3,888
   yes MOEX: 3,051
   yes ELFV: 2,583
   yes CHGZ: 2,343
   yes AVAN: 1,939
   yes ALRS: 1,050
   yes WTCM: 810
   yes OGKB: 643
   yes APTK: 615
   yes PIKK: 346


In [5]:
analyzer.analyze_news()
None

БАЗОВАЯ СТАТИСТИКА:
Всего новостей: 91,955
Уникальных заголовков: 43,770
Новости с тикерами (после маппинга): 74,164 (80.7%)
АНАЛИЗ ИСТОЧНИКОВ:
smart_lab: 34,517 (37.5%)
finam: 22,433 (24.4%)
bcs: 12,424 (13.5%)
bcs_tech: 11,560 (12.6%)
rdv: 5,721 (6.2%)
t_invest: 4,498 (4.9%)
t_analytic: 802 (0.9%)
ВРЕМЕННОЙ АНАЛИЗ:
Период: 2022-05-26 13:32:40 -> 2024-12-15 13:41:06
Распределение по дням недели:
   Monday: 2,152 (19.5%)
   Tuesday: 1,974 (17.9%)
   Wednesday: 1,980 (18.0%)
   Thursday: 2,048 (18.6%)
   Friday: 2,273 (20.6%)
   Saturday: 396 (3.6%)
   Sunday: 198 (1.8%)
Пик публикаций: 6:00
АНАЛИЗ ДЛИНЫ ТЕКСТОВ:
Заголовок - среднее: 8, медиана: 8
Тело - среднее: 979, медиана: 719


In [6]:
analyzer.analyze_candles_availability()
None

СТАТИСТИКА:
Уникальных тикеров в новостях: 24
Тикеров с ценовыми данными: 23 (95.8%)
Новостей с тикерами: 347,529
Новостей с доступными ценами: 306,842 (88.3%)
ТОП-20 ТИКЕРОВ С ЦЕНОВЫМИ ДАННЫМИ:
   ABIO: 73,475 новостей
   AFKS: 58,974 новостей
   AFLT: 47,794 новостей
   AKRN: 42,989 новостей
   ABRD: 32,222 новостей
   CARM: 10,562 новостей
   APRI: 10,433 новостей
   BANE: 6,893 новостей
   AQUA: 5,895 новостей
   BRZL: 3,888 новостей
   MOEX: 3,051 новостей
   ELFV: 2,583 новостей
   CHGZ: 2,343 новостей
   AVAN: 1,939 новостей
   ALRS: 1,050 новостей
   WTCM: 810 новостей
   OGKB: 643 новостей
   APTK: 615 новостей
   PIKK: 346 новостей
   MRKV: 193 новостей


In [7]:
analyzer.analyze_labeled_data()
None

OLD РАЗМЕТКА:
Всего записей: 79,705
GPT4O РАЗМЕТКА:
Всего записей: 15,000
АНАЛИЗ ТОНАЛЬНОСТИ:
   Среднее: 0.156
   Медиана: 0.000
   Позитивные: 5792 (38.6%)
   Негативные: 2882 (19.2%)
   Нейтральные: 6326 (42.2%)
LLAMA3 РАЗМЕТКА:
Всего записей: 24,398
АНАЛИЗ ТОНАЛЬНОСТИ:
   Среднее: 0.164
   Медиана: 0.000
   Позитивные: 9839 (40.3%)
   Негативные: 5230 (21.4%)
   Нейтральные: 9329 (38.2%)


In [8]:
analyzer.generate_report()
None

ДАННЫЕ ДЛЯ РАЗМЕТКИ:
Всего новостей с тикерами: 347,529
ЦЕНОВЫЕ ДАННЫЕ:
Тикеров с ценами: 23
Новостей с ценами: 306,842


# Получение итогого датасета

In [9]:
DATA_PATH = "./RussianFinancialNews"

HORIZON_MIN = 10
THRESHOLD = 0.01

TIMEFRAME = "10m"

SAVE_PATH = "prepared_dataset.parquet"

In [10]:
import pandas as pd
from pathlib import Path

In [11]:
data_path = Path(DATA_PATH)

news = pd.read_parquet(data_path / "news_collection.parquet")

print(len(news))
news.head()

91955


,title,body,date,time,tags,source
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,16:16:54,['цифры'],rdv
1,no title,""" Рейтинг акций комьюнити РДВ. #опрос """,2024-09-09,15:38:21,['опрос'],rdv
2,no title,"""Нефть Brent упала до $71, Urals - до $60 за б...",2024-09-09,14:40:17,['шокновости'],rdv
3,no title,"""️ Ставка ЦБ вряд ли уйдёт выше 18%. ЦБ переже...",2024-09-09,12:54:10,['VIPtalk'],rdv
4,no title,"""Эксперты клуба RDV PREMIUM 361° дали коммента...",2024-09-09,10:25:07,[],rdv


In [12]:
news.rename(columns={'body': 'text'}, inplace=True)

news["datetime"] = pd.to_datetime(
    news["date"] + " " + news["time"],
    errors="coerce"
)

news = news.dropna(subset=["datetime"])

print(len(news))
news.head()

11021


,title,text,date,time,tags,source,datetime
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,16:16:54,['цифры'],rdv,2024-09-09 16:16:54
1,no title,""" Рейтинг акций комьюнити РДВ. #опрос """,2024-09-09,15:38:21,['опрос'],rdv,2024-09-09 15:38:21
2,no title,"""Нефть Brent упала до $71, Urals - до $60 за б...",2024-09-09,14:40:17,['шокновости'],rdv,2024-09-09 14:40:17
3,no title,"""️ Ставка ЦБ вряд ли уйдёт выше 18%. ЦБ переже...",2024-09-09,12:54:10,['VIPtalk'],rdv,2024-09-09 12:54:10
4,no title,"""Эксперты клуба RDV PREMIUM 361° дали коммента...",2024-09-09,10:25:07,[],rdv,2024-09-09 10:25:07


In [13]:
import json

with open("./data/ticker_mapping_full.json", "r", encoding="utf-8") as f:
    name_to_ticker = json.load(f)

def map_to_ticker(tag):
    if not tag:
        return None
    
    if tag in name_to_ticker:
        return name_to_ticker[tag]
    
    tag_clean = tag.replace(' ', '').replace('-', '').upper()
    for name, ticker in name_to_ticker.items():
        name_clean = name.replace(' ', '').replace('-', '').upper()
        if tag_clean == name_clean or tag_clean in name_clean or name_clean in tag_clean:
            return ticker
    
    return None

def convert_tags(tags):
    if not tags:
        return []
    result = []
    for tag in tags:
        ticker = map_to_ticker(tag)
        if ticker:
            result.append(ticker)
    return list(set(result))

news["tickers"] = news["tags"].apply(convert_tags)
news["has_ticker"] = news["tickers"].apply(len) > 0

news = news[news["has_ticker"]].copy()

print(len(news))

8430


In [14]:
news = news.explode("tickers").rename(columns={"tickers": "ticker"})

print(len(news))
news.head()

26552


,title,text,date,time,tags,source,datetime,ticker,has_ticker
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,16:16:54,['цифры'],rdv,2024-09-09 16:16:54,ALRS,True
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,16:16:54,['цифры'],rdv,2024-09-09 16:16:54,AFKS,True
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,16:16:54,['цифры'],rdv,2024-09-09 16:16:54,APRI,True
0,no title,"""После падения добычи в 2022-2023 годах Газпро...",2024-09-09,16:16:54,['цифры'],rdv,2024-09-09 16:16:54,ABIO,True
1,no title,""" Рейтинг акций комьюнити РДВ. #опрос """,2024-09-09,15:38:21,['опрос'],rdv,2024-09-09 15:38:21,ABRD,True


In [15]:
ticker_counts = news["ticker"].value_counts()
top_tickers = ticker_counts.index.tolist()

print(top_tickers)

news = news[news["ticker"].isin(top_tickers)].copy()

['ABIO', 'ABRD', 'AFKS', 'APRI', 'AFLT', 'CARM', 'ALRS', 'AKRN', 'BLNG', 'BANE', 'AVAN', 'CHGZ', 'WTCM', 'BRZL', 'MOEX', 'AQUA', 'APTK', 'OGKB', 'JNOS', 'ZILL', 'PIKK', 'ELFV']


In [16]:
close = pd.read_csv(data_path / "candles_data" / TIMEFRAME / "close.csv")

close["begin"] = pd.to_datetime(close["begin"])

close.head()

,begin,ABIO,ABRD,AFKS,AFLT,AKRN,ALRS,AMEZ,APTK,AQUA,...,WUSH,YAKG,YKEN,YKENP,YNDX,YRSB,YRSBP,ZAYM,ZILL,ZVEZ
0,2022-07-01 09:50:00,57.78,NaN,14.310,26.60,18182.0,67.49,19.855,12.366,499.0,...,NaN,110.00,NaN,NaN,1595.2,NaN,NaN,NaN,NaN,NaN
1,2022-07-01 10:00:00,56.50,172.0,14.501,26.16,18548.0,66.09,19.850,12.368,498.5,...,NaN,108.25,NaN,NaN,1572.2,NaN,NaN,NaN,2950.0,NaN
2,2022-07-01 10:10:00,56.46,172.0,14.755,26.70,18450.0,66.16,20.675,12.362,501.0,...,NaN,106.50,0.2775,NaN,1582.6,NaN,NaN,NaN,2900.0,3.545
3,2022-07-01 10:20:00,56.44,171.0,14.810,26.92,18342.0,66.18,20.005,12.424,502.5,...,NaN,105.15,NaN,NaN,1577.4,NaN,83.0,NaN,2900.0,NaN
4,2022-07-01 10:30:00,56.10,171.5,14.642,26.84,18400.0,66.10,19.885,12.394,503.0,...,NaN,105.05,NaN,NaN,1561.2,NaN,83.0,NaN,2890.0,NaN


In [17]:
close_long = close.melt(
    id_vars="begin",
    var_name="ticker",
    value_name="close"
)

close_long = close_long.sort_values(["begin", "ticker"]).reset_index(drop=True)

In [18]:
news = news.sort_values(["datetime", "ticker"]).reset_index(drop=True)

price_t = pd.merge_asof(
    news,
    close_long,
    left_on="datetime",
    right_on="begin",
    by="ticker",
    direction="backward"
)

price_t = price_t.rename(columns={
    "close": "price_t",
    "begin": "time_t"
})

price_t.head()

,title,text,date,time,tags,source,datetime,ticker,has_ticker,time_t,price_t
0,no title,"""AGCO: финансовое состояние хорошее, спрос уст...",2022-05-26,13:32:40,['идея'],t_analytic,2022-05-26 13:32:40,ABIO,True,NaT,NaN
1,no title,"""AGCO: финансовое состояние хорошее, спрос уст...",2022-05-26,13:32:40,['идея'],t_analytic,2022-05-26 13:32:40,ABRD,True,NaT,NaN
2,no title,"""AGCO: финансовое состояние хорошее, спрос уст...",2022-05-26,13:32:40,['идея'],t_analytic,2022-05-26 13:32:40,AFKS,True,NaT,NaN
3,no title,"""​​Сегежа выигрывает от слабого рубля или нет?...",2022-05-27,09:04:08,['россия'],t_analytic,2022-05-27 09:04:08,ABIO,True,NaT,NaN
4,no title,"""​​Сегежа выигрывает от слабого рубля или нет?...",2022-05-27,09:04:08,['россия'],t_analytic,2022-05-27 09:04:08,AFKS,True,NaT,NaN


In [19]:
price_t["target_time"] = price_t["datetime"] + pd.Timedelta(minutes=HORIZON_MIN)

close_long_sorted = close_long.sort_values(["begin", "ticker"])

price_tH = pd.merge_asof(
    price_t.sort_values(["target_time", "ticker"]),
    close_long_sorted,
    left_on="target_time",
    right_on="begin",
    by="ticker",
    direction="forward"
)

price_tH = price_tH.rename(columns={
    "close": "price_t_H",
    "begin": "time_t_H"
})

price_tH.head()

,title,text,date,time,tags,source,datetime,ticker,has_ticker,time_t,price_t,target_time,time_t_H,price_t_H
0,no title,"""AGCO: финансовое состояние хорошее, спрос уст...",2022-05-26,13:32:40,['идея'],t_analytic,2022-05-26 13:32:40,ABIO,True,NaT,NaN,2022-05-26 13:42:40,2022-07-01 09:50:00,57.78
1,no title,"""AGCO: финансовое состояние хорошее, спрос уст...",2022-05-26,13:32:40,['идея'],t_analytic,2022-05-26 13:32:40,ABRD,True,NaT,NaN,2022-05-26 13:42:40,2022-07-01 09:50:00,NaN
2,no title,"""AGCO: финансовое состояние хорошее, спрос уст...",2022-05-26,13:32:40,['идея'],t_analytic,2022-05-26 13:32:40,AFKS,True,NaT,NaN,2022-05-26 13:42:40,2022-07-01 09:50:00,14.31
3,no title,"""​​Сегежа выигрывает от слабого рубля или нет?...",2022-05-27,09:04:08,['россия'],t_analytic,2022-05-27 09:04:08,ABIO,True,NaT,NaN,2022-05-27 09:14:08,2022-07-01 09:50:00,57.78
4,no title,"""​​Сегежа выигрывает от слабого рубля или нет?...",2022-05-27,09:04:08,['россия'],t_analytic,2022-05-27 09:04:08,AFKS,True,NaT,NaN,2022-05-27 09:14:08,2022-07-01 09:50:00,14.31


In [20]:
df = price_tH.copy()

# нет цен
df = df.dropna(subset=["price_t", "price_t_H"])

# удаляем переходы на следующий день
df["same_day"] = (
    df["time_t"].dt.date == df["time_t_H"].dt.date
)

df = df[df["same_day"]]

print(len(df))

11202


In [21]:
df["return"] = (df["price_t_H"] - df["price_t"]) / df["price_t"]

In [22]:
def get_label(r):
    if r > THRESHOLD:
        return 1
    elif r < -THRESHOLD:
        return 2
    else:
        return 0

df["label"] = df["return"].apply(get_label)

In [23]:
final_df = df[
    [
        "datetime",
        "ticker",
        "text",
        "price_t",
        "price_t_H",
        "return",
        "label"
    ]
].reset_index(drop=True)

final_df.head()

,datetime,ticker,text,price_t,price_t_H,return,label
0,2022-07-01 10:03:32,ABIO,""" Ситуация с Газпромом (GAZP) подсветила важне...",56.500,56.44,-0.001062,0
1,2022-07-01 10:03:32,ABRD,""" Ситуация с Газпромом (GAZP) подсветила важне...",172.000,171.00,-0.005814,0
2,2022-07-01 10:03:32,AFKS,""" Ситуация с Газпромом (GAZP) подсветила важне...",14.501,14.81,0.021309,1
3,2022-07-01 10:03:32,AFLT,""" Ситуация с Газпромом (GAZP) подсветила важне...",26.160,26.92,0.029052,1
4,2022-07-01 10:03:32,AKRN,""" Ситуация с Газпромом (GAZP) подсветила важне...",18548.000,18342.00,-0.011106,2


In [24]:
final_df.to_parquet(SAVE_PATH)

print("Saved:", SAVE_PATH)
print("Shape:", final_df.shape)

Saved: prepared_dataset.parquet
Shape: (11202, 7)


In [25]:
final_df.label.value_counts()

label
0    10636
1      289
2      277
Name: count, dtype: int64

In [26]:
final_df.head(20)

,datetime,ticker,text,price_t,price_t_H,return,label
0,2022-07-01 10:03:32,ABIO,""" Ситуация с Газпромом (GAZP) подсветила важне...",56.500,56.440,-0.001062,0
1,2022-07-01 10:03:32,ABRD,""" Ситуация с Газпромом (GAZP) подсветила важне...",172.000,171.000,-0.005814,0
2,2022-07-01 10:03:32,AFKS,""" Ситуация с Газпромом (GAZP) подсветила важне...",14.501,14.810,0.021309,1
3,2022-07-01 10:03:32,AFLT,""" Ситуация с Газпромом (GAZP) подсветила важне...",26.160,26.920,0.029052,1
4,2022-07-01 10:03:32,AKRN,""" Ситуация с Газпромом (GAZP) подсветила важне...",18548.000,18342.000,-0.011106,2
5,2022-07-01 10:03:32,BLNG,""" Ситуация с Газпромом (GAZP) подсветила важне...",10.020,10.015,-0.000499,0
6,2022-07-01 10:46:40,ABIO,"""Защитные акции не защитили С начала года и...",56.100,56.360,0.004635,0
7,2022-07-01 10:46:40,ABRD,"""Защитные акции не защитили С начала года и...",171.500,170.500,-0.005831,0
8,2022-07-01 16:00:03,ABIO,"""​​Взлеты и падения: китайские драконы летят в...",56.920,56.800,-0.002108,0
9,2022-07-01 16:00:03,ABRD,"""​​Взлеты и падения: китайские драконы летят в...",172.000,172.000,0.000000,0
